In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_type", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", StringType(), True),
    StructField("city", StringType(), True)
])

In [0]:
# import requests
# import json
# response= requests.get("https://holmes-hampton-going-busy.trycloudflare.com/event")
# data = response.json()
# with open("/Volumes/file_upload/files/tmp/delta/user_event_checkpoint/raw_landing/data.json", "w") as f:
#     json.dump(data,f)
# print(response.status_code)
# print(response.json())

In [0]:
# import requests
# import json
# import time
# import os
# import schedule

# API_URL = "https://creation-enjoy-builds-heating.trycloudflare.com/event"
# OUTPUT_PATH_DIR = "/Volumes/file_upload/files/tmp/delta/user_event_checkpoint/raw_landing/"

# def fetch_event():
#     response = requests.get(API_URL,timeout=10)
#     if response.status_code == 200:
#         data = response.json()
#         file_name = f"event_{int(time.time() * 1000)}.json"
#         file_path = os.path.join(
#             OUTPUT_PATH_DIR,file_name
#         )
#         with open(file_path,"w") as f:
#             json.dump(data,f)
#         # print(data)
#     else:
#         print(response.status_code)
# schedule.every(5).seconds.do(fetch_event)
# while True:
#     schedule.run_pending()
#     time.sleep(1)

In [0]:
bronze_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format","json") \
    .option("cloudFiles.schemaLocation","/Volumes/file_upload/files/tmp/delta/bronze_event_checkpoint/bronze_schema") \
    .schema(event_schema) \
    .load("/Volumes/file_upload/files/tmp/delta/user_event_checkpoint/raw_landing/")

In [0]:
bronze_df = bronze_df.withColumn(
    "ingestion_time",
    current_timestamp()
)

In [0]:
bronze_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/file_upload/files/tmp/delta/bronze_event_checkpoint/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce.bronze.bronze_events")